# 06 Baseline Models

This notebook trains the structured-data baselines used as reference points for the multimodal study. It evaluates Logistic Regression, Random Forest, and XGBoost across the 6h, 12h, and 24h prediction horizons using the leakage-safe datasets from feature engineering.

## Baseline strategy

- Convert each horizon trajectory dataset into one stay-level tabular example.
- Aggregate temporal features with `mean`, `min`, `max`, and `last`.
- Train only on the train split and report validation/test metrics separately.
- Save both metrics and per-stay prediction tables for downstream comparison.

In [1]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'src').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime import load_project_runtime

runtime = load_project_runtime(start=PROJECT_ROOT)
IN_COLAB = runtime.in_colab
PROJECT_ROOT = runtime.project_root
config = runtime.config
paths = runtime.paths

PROJECT_ROOT

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main')

In [2]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy scikit-learn xgboost

In [3]:
import pandas as pd

from src.evaluation.metrics import compute_binary_classification_metrics
from src.models.baselines import (
    build_baseline_models,
    fit_and_predict_baseline,
    make_stay_level_tabular_dataset,
    split_tabular_dataset,
)
from src.utils.config import load_config
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.utils.paths import ensure_directories, resolve_project_paths

In [4]:
config['baselines']

{'models': ['logistic_regression', 'random_forest', 'xgboost'],
 'tabular_aggregations': ['mean', 'min', 'max', 'last'],
 'imputation_strategy': 'median',
 'random_state': 42,
 'logistic_regression': {'max_iter': 1000, 'class_weight': 'balanced'},
 'random_forest': {'n_estimators': 300,
  'max_depth': 12,
  'min_samples_leaf': 5,
  'class_weight': 'balanced_subsample'},
 'xgboost': {'n_estimators': 300,
  'max_depth': 6,
  'learning_rate': 0.05,
  'subsample': 0.8,
  'colsample_bytree': 0.8}}

## Load horizon datasets

In [5]:
horizon_tables = {}
for horizon in config['prediction']['horizons_hours']:
    dataset_name = f'horizon_{int(horizon)}h'
    path = paths['processed_data_dir'] / '04_feature_engineering' / f'{dataset_name}.csv'
    assert path.exists(), f'Missing horizon dataset: {dataset_name}. Run 04_feature_engineering first.'
    horizon_tables[dataset_name] = pd.read_csv(path, parse_dates=['hour', 'prediction_time', 'INTIME', 'OUTTIME'], low_memory=False)

{key: value.shape for key, value in horizon_tables.items()}

{'horizon_6h': (1268294, 90),
 'horizon_12h': (1131670, 90),
 'horizon_24h': (860480, 90)}

## Train baselines horizon-by-horizon

In [6]:
models = build_baseline_models(config)
results_rows = []
artifact_bundle = {}

for dataset_name, horizon_df in horizon_tables.items():
    tabular_df = make_stay_level_tabular_dataset(
        horizon_df=horizon_df,
        aggregations=config['baselines']['tabular_aggregations'],
    )
    artifact_bundle[f'{dataset_name}_tabular'] = tabular_df
    splits = split_tabular_dataset(tabular_df)
    train_X, train_y = splits['train']
    val_X, val_y = splits['val']
    test_X, test_y = splits['test']

    for model_name, model in models.items():
        if train_X.empty or test_X.empty:
            continue

        test_prob, feature_cols = fit_and_predict_baseline(model, train_X, train_y, test_X)
        test_metrics = compute_binary_classification_metrics(test_y, test_prob)
        results_rows.append({
            'dataset_name': dataset_name,
            'split': 'test',
            'model_name': model_name,
            **test_metrics,
            'n_features': len(feature_cols),
            'n_examples': int(len(test_y)),
        })

        test_predictions = test_X[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID']].copy()
        test_predictions['y_true'] = test_y.to_numpy()
        test_predictions['y_prob'] = test_prob
        test_predictions['dataset_name'] = dataset_name
        test_predictions['model_name'] = model_name
        artifact_bundle[f'{dataset_name}_{model_name}_test_predictions'] = test_predictions

        if not val_X.empty:
            val_prob, _ = fit_and_predict_baseline(model, train_X, train_y, val_X)
            val_metrics = compute_binary_classification_metrics(val_y, val_prob)
            results_rows.append({
                'dataset_name': dataset_name,
                'split': 'val',
                'model_name': model_name,
                **val_metrics,
                'n_features': len(feature_cols),
                'n_examples': int(len(val_y)),
            })

results_df = pd.DataFrame(results_rows).sort_values(['dataset_name', 'split', 'model_name']).reset_index(drop=True)
results_df

,dataset_name,split,model_name,auroc,auprc,precision,recall,f1,brier_score,specificity,sensitivity,tp,fp,tn,fn,n_features,n_examples
0,horizon_12h,test,logistic_regression,0.853810,0.265201,0.152678,0.799163,0.256376,0.151014,0.799546,0.799163,191,1060,4228,48,304,5527
1,horizon_12h,test,random_forest,0.998746,0.980104,0.983051,0.970711,0.976842,0.006747,0.999244,0.970711,232,4,5284,7,304,5527
2,horizon_12h,test,xgboost,0.998611,0.961338,0.931174,0.962343,0.946502,0.004433,0.996785,0.962343,230,17,5271,9,304,5527
3,horizon_12h,val,logistic_regression,0.827959,0.183266,0.122400,0.753695,0.210599,0.152647,0.790168,0.753695,153,1097,4131,50,304,5431
4,horizon_12h,val,random_forest,0.994672,0.965877,0.960591,0.960591,0.960591,0.006871,0.998470,0.960591,195,8,5220,8,304,5431
5,horizon_12h,val,xgboost,0.998931,0.968097,0.926829,0.935961,0.931373,0.004095,0.997131,0.935961,190,15,5213,13,304,5431
6,horizon_24h,test,logistic_regression,0.897076,0.339245,0.179975,0.811429,0.294606,0.116882,0.831642,0.811429,142,647,3196,33,304,4018
7,horizon_24h,test,random_forest,0.998785,0.975328,0.982955,0.988571,0.985755,0.005878,0.999219,0.988571,173,3,3840,2,304,4018
8,horizon_24h,test,xgboost,0.998962,0.965150,0.929348,0.977143,0.952646,0.003374,0.996617,0.977143,171,13,3830,4,304,4018
9,horizon_24h,val,logistic_regression,0.870140,0.239899,0.141148,0.808219,0.240326,0.124630,0.813555,0.808219,118,718,3133,28,304,3997


## Inspect best-performing baselines

In [7]:
results_df.sort_values(['split', 'auprc'], ascending=[True, False]).head(12) if not results_df.empty else results_df

,dataset_name,split,model_name,auroc,auprc,precision,recall,f1,brier_score,specificity,sensitivity,tp,fp,tn,fn,n_features,n_examples
1,horizon_12h,test,random_forest,0.998746,0.980104,0.983051,0.970711,0.976842,0.006747,0.999244,0.970711,232,4,5284,7,304,5527
13,horizon_6h,test,random_forest,0.997992,0.976293,0.982993,0.966555,0.974705,0.007575,0.999096,0.966555,289,5,5523,10,304,5827
7,horizon_24h,test,random_forest,0.998785,0.975328,0.982955,0.988571,0.985755,0.005878,0.999219,0.988571,173,3,3840,2,304,4018
14,horizon_6h,test,xgboost,0.998947,0.973890,0.939297,0.983278,0.960784,0.003763,0.996563,0.983278,294,19,5509,5,304,5827
8,horizon_24h,test,xgboost,0.998962,0.965150,0.929348,0.977143,0.952646,0.003374,0.996617,0.977143,171,13,3830,4,304,4018
2,horizon_12h,test,xgboost,0.998611,0.961338,0.931174,0.962343,0.946502,0.004433,0.996785,0.962343,230,17,5271,9,304,5527
6,horizon_24h,test,logistic_regression,0.897076,0.339245,0.179975,0.811429,0.294606,0.116882,0.831642,0.811429,142,647,3196,33,304,4018
0,horizon_12h,test,logistic_regression,0.853810,0.265201,0.152678,0.799163,0.256376,0.151014,0.799546,0.799163,191,1060,4228,48,304,5527
12,horizon_6h,test,logistic_regression,0.844136,0.243081,0.165198,0.752508,0.270921,0.154118,0.794320,0.752508,225,1137,4391,74,304,5827
17,horizon_6h,val,xgboost,0.998931,0.976774,0.946809,0.930314,0.938489,0.004600,0.997252,0.930314,267,15,5443,20,304,5745


## Save metrics and prediction artifacts

In [8]:
output_dir = paths['processed_data_dir'] / '06_baseline_models'
artifact_bundle['baseline_results'] = results_df
saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

{'horizon_6h_tabular': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/06_baseline_models/horizon_6h_tabular.csv',
 'horizon_6h_logistic_regression_test_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/06_baseline_models/horizon_6h_logistic_regression_test_predictions.csv',
 'horizon_6h_random_forest_test_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/06_baseline_models/horizon_6h_random_forest_test_predictions.csv',
 'horizon_6h_xgboost_test_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/06_baseline_models/horizon_6h_xgboost_test_predictions.csv',
 'horizon_12h_tabular': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/06_baseline_models/horizon_12h_tabular.csv',
 'horizon_12h_logistic_regression_test_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/06_baseline_models/horizon_12h_logistic_

In [9]:
manifest_path = paths['manifests_dir'] / '06_baseline_models_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='06_baseline_models',
    config=config,
    extra={
        'saved_artifacts': saved_paths,
        'models_run': list(models.keys()),
        'result_rows': int(len(results_df)),
    },
)
manifest_path

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/manifests/06_baseline_models_manifest.json')

## Review checklist

Before moving to multimodal models, review:

- Which baseline is strongest at each horizon?
- Does AUPRC drop sharply from 6h to 24h?
- Are validation and test results reasonably aligned?
- Does XGBoost materially outperform simpler models?
- Are any horizons too imbalanced for stable comparison?

## Next notebook

`07_multimodal_models.ipynb` will introduce the deep structured encoders, text encoders, and fusion strategies that should be compared against these baselines.